# Lamoda · группы клиентов и последовательности покупок
**Задача.** Оценить вероятность покупки за три месяца и отобрать верхнюю пятую часть клиентов. Ноутбук сначала строит признаки предыдущего решения, затем сравнивает модели по категориям и сериям заказов. Итоговый файл — `submission_groups.csv`.

Запустите все ячейки из корня репозитория с исходным архивом конкурса рядом с ноутбуком или задайте `LAMODA_DATA`. Нужны Python 3.12, NumPy, pandas, PyArrow, scikit-learn и LightGBM. После каждого дорогостоящего обучения прогноз атомарно сохраняется в локальный каталог `.checkpoints/`. Повторный запуск использует совпадающие контрольные точки.

In [ ]:
from pathlib import Path
import os
from zipfile import ZipFile

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from threadpoolctl import threadpool_limits

slug = "lamoda-hse-repeat-purchase-2026"
files = ["train.csv", "test.csv", "orders.parquet", "products.parquet", "sample_submission.csv"]
candidates = [Path("data"), Path("."), Path(slug + ".zip"),
              Path("/kaggle/input") / slug, Path("/kaggle/input/competitions") / slug]
attachments = Path("/workspace/attachments")
if attachments.exists():
    candidates.extend(sorted(attachments.glob("*/" + slug + ".zip")))
if "LAMODA_DATA" in os.environ:
    candidates = [Path(os.environ["LAMODA_DATA"])]
data = next((p for p in candidates if (p.is_file() and p.suffix == ".zip")
             or all((p / name).is_file() for name in files)), None)
if data is None:
    raise FileNotFoundError("Укажите путь к данным в LAMODA_DATA.")

def read_table(name, **kwargs):
    reader = pd.read_csv if name.endswith(".csv") else pd.read_parquet
    if data.is_dir():
        return reader(data / name, **kwargs)
    with ZipFile(data) as archive, archive.open(name) as stream:
        return reader(stream, **kwargs)

train = read_table("train.csv", parse_dates=["scoring_date"])
test = read_table("test.csv", parse_dates=["scoring_date"])
sample = read_table("sample_submission.csv")
orders = read_table("orders.parquet")
products = read_table("products.parquet", columns=["product_id", "brand", "category", "category2", "gender_product"])
orders["order_date"] = pd.to_datetime(orders["order_date"])
orders = orders.merge(products, how="left", on="product_id", validate="many_to_one")
assert train.target.isin([0, 1]).all() and test.customer_id.is_unique
print(train.groupby("scoring_date").target.agg(["size", "mean"]).to_string())

### История клиента на дату прогноза
Строка истории соответствует товарной позиции. Считаем количество отдельных заказов, интервалы между днями покупок, суммы, скидки и разнообразие ассортимента. Окна в 30, 90, 180 и 365 дней отражают изменения активности. Там, где история отсутствует, бустинг сам обрабатывает пропуски.

In [ ]:
def features(cohort):
    blocks = []
    for cutoff, customers in cohort.groupby("scoring_date", sort=False):
        history = orders.loc[
            orders.customer_id.isin(customers.customer_id) & (orders.order_date < cutoff)
        ].sort_values(["order_date", "order_id"], kind="stable")
        by = history.groupby("customer_id")
        stats = by.agg(
            first=("order_date", "min"), last=("order_date", "max"),
            n_orders=("order_id", "nunique"), n_days=("order_date", "nunique"),
            n_items=("product_id", "size"), n_brands=("brand", "nunique"),
            n_categories=("category", "nunique"), total_paid=("paid_price", "sum"),
            mean_price=("paid_price", "mean"), mean_discount=("discount_percent", "mean"),
            mean_loyalty=("loyalty_percent", "mean"), last_loyalty=("loyalty_percent", "last"),
        )
        stats["recency"] = (cutoff - stats.pop("last")).dt.days
        stats["age_days"] = (cutoff - stats.pop("first")).dt.days
        stats["mean_basket"] = stats.total_paid / stats.n_orders
        days = history[["customer_id", "order_date"]].drop_duplicates()
        gaps = days.groupby("customer_id").order_date.diff().dt.days
        stats = stats.join(gaps.groupby(days.customer_id).agg(["mean", "std", "last"]).add_prefix("gap_"))
        stats["recency_over_gap"] = stats.recency / stats.gap_mean
        for window in (30, 90, 180, 365):
            recent = history.loc[history.order_date >= cutoff - pd.Timedelta(days=window)]
            totals = recent.groupby("customer_id").agg(
                orders=("order_id", "nunique"), spend=("paid_price", "sum"),
                items=("product_id", "size"),
            )
            stats = stats.join(totals.reindex(stats.index, fill_value=0).add_suffix(f"_{window}"))
        block = stats.reindex(customers.customer_id)
        block.index = customers.index
        blocks.append(block)
    return pd.concat(blocks).reindex(cohort.index)

X = features(train)
X_test = features(test).loc[:, X.columns]
y = train.target
print("Матрицы признаков:", X.shape, X_test.shape)

### Метрика и базовый классификатор
AP@20% учитывает первые 5 000 из 25 000 клиентов на каждом контрольном срезе.

In [ ]:
def ap20(labels, scores):
    k = max(1, int(np.ceil(0.2 * len(labels))))
    ranked = np.asarray(labels)[np.argsort(-np.asarray(scores), kind="stable")[:k]]
    precision = ranked.cumsum() / np.arange(1, k + 1)
    return (precision * ranked).sum() / max(1, min(k, labels.sum()))

def fit_model(features, labels):
    model = HistGradientBoostingClassifier(max_iter=180, learning_rate=0.055,
            max_leaf_nodes=23, min_samples_leaf=60, l2_regularization=3,
            early_stopping=False, random_state=42)
    with threadpool_limits(limits=4):
        model.fit(features, labels)
    return model



### Сезонность, интенсивность и ассортимент
Сравниваем исходную модель, модель с динамикой и модель со всеми признаками. Для каждого контрольного месяца используем метки предыдущих когорт.

In [ ]:
def rich_features(cohort):
    blocks = []
    for cutoff, clients in cohort.groupby("scoring_date"):
        h = orders.loc[orders.customer_id.isin(clients.customer_id) & (orders.order_date < cutoff)].copy()
        h["age"] = (cutoff - h.order_date).dt.days
        h = h.sort_values("age", kind="stable")
        days = h.drop_duplicates(["customer_id", "order_date"]).copy()
        g = days.groupby("customer_id")
        days["gap"] = g.age.diff()
        parts = [g.gap.agg(["median", "min", "max"]).add_prefix("interval_")]
        for window in (45, 90, 180, 365):
            recent = days[days.age.between(14, 13 + window)]
            counts = recent.groupby("customer_id").size()
            previous = days[days.age.between(14 + window, 13 + 2 * window)].groupby("customer_id").size()
            parts.append(((counts.reindex(clients.customer_id, fill_value=0) + 1)
                          / (previous.reindex(clients.customer_id, fill_value=0) + 1)).rename(f"trend_{window}"))
        for lag in (365, 730):
            season = days[days.age.between(lag - 91, lag)]
            parts.append(season.groupby("customer_id").size().rename(f"prior_season_{lag}"))
        for period in (30, 90):
            parts.append(days.assign(bucket=days.age // period).groupby("customer_id").bucket.nunique().rename(f"active_periods_{period}"))
        for col in ["category", "gender_product"]:
            parts.append(pd.crosstab(h.customer_id, h[col], normalize="index").add_prefix(f"mix_{col}_"))
        for col in ["loyalty_percent", "discount_percent", "paid_price"]:
            parts.append(h.groupby("customer_id")[col].agg(["max", "min", "std", "first"]).add_prefix(col + "_"))
        parts.append(h.groupby("customer_id").agg(unique_sku=("product_id","nunique"), subcats=("category2","nunique")))
        block = pd.concat(parts, axis=1).reindex(clients.customer_id)
        block.index = clients.index
        blocks.append(block)
    return pd.concat(blocks).reindex(cohort.index)

extra = rich_features(train)
extra_test = rich_features(test).reindex(columns=extra.columns)

In [ ]:
def rhythm_features(cohort):
    result = []
    for cutoff, clients in cohort.groupby("scoring_date"):
        h = orders.loc[orders.customer_id.isin(clients.customer_id) & (orders.order_date < cutoff)]
        baskets = h.groupby(["customer_id", "order_id"]).agg(
            date=("order_date", "min"), amount=("paid_price", "sum"), units=("product_id", "size"),
            discount=("discount_percent", "mean"), loyalty=("loyalty_percent", "mean")
        ).reset_index().sort_values(["customer_id", "date", "order_id"], ascending=[True, False, False])
        baskets["age"] = (cutoff - baskets.date).dt.days
        parts = {}
        for unit, visits in [("day", baskets.drop_duplicates(["customer_id", "date"])), ("order", baskets)]:
            visits = visits.copy()
            g = visits.groupby("customer_id")
            rec = g.age.first()
            visits["gap"] = g.age.diff()
            for scope, recent in [("all", visits), ("3", g.head(4)), ("6", g.head(7)),
                                  ("year", visits[visits.age <= 379])]:
                grouped = recent.groupby("customer_id")
                for stat in ["mean", "median"]:
                    period = grouped.gap.agg(stat).replace(0, np.nan)
                    prefix = f"rhythm_{unit}_{scope}_{stat}"
                    parts[prefix] = rec / period
                    parts[prefix + "_delta"] = rec - period
                    parts[prefix + "_net"] = (rec - 14) / period
                parts[f"rhythm_{unit}_{scope}_cv"] = grouped.gap.std() / grouped.gap.mean().replace(0, np.nan)
        g = baskets.groupby("customer_id")
        for field in ["amount", "units", "loyalty", "discount"]:
            avg = g[field].mean()
            parts["basket_" + field] = avg
            parts["basket_last_" + field] = g[field].first() / avg.replace(0, np.nan)
        block = pd.DataFrame(parts).reindex(clients.customer_id)
        block.index = clients.index
        result.append(block)
    return pd.concat(result).reindex(cohort.index)

extra = extra.join(rhythm_features(train))
extra_test = extra_test.join(rhythm_features(test))

In [ ]:
def cycle_features(cohort):
    blocks = []
    for cutoff, customers in cohort.groupby("scoring_date"):
        h = orders[orders.customer_id.isin(customers.customer_id) & (orders.order_date < cutoff)].copy()
        h["age"] = (cutoff - h.order_date).dt.days
        duration = ((cutoff + pd.DateOffset(months=3)) - cutoff).days
        parts = {}
        for key in ("customer", "category", "category2", "brand", "gender_product", "product_id"):
            group = ["customer_id"] if key == "customer" else ["customer_id", key]
            visits = h[group + ["age"]].drop_duplicates().sort_values("age")
            g = visits.groupby(group)
            visits["interval"] = g.age.diff()
            info = g.agg(age=("age", "first"), n=("age", "size"), last=("interval", "first"),
                         median=("interval", "median"), mean=("interval", "mean"),
                         std=("interval", "std"))
            for rule in ("median", "last", "mean"):
                due = info[rule] - info.age
                score = (due - duration/2).abs()
                record = pd.DataFrame({"center": score, "due": due, "n": info.n, "std": info["std"]})
                if key == "customer":
                    parts["cycle_" + rule] = due
                else:
                    label = f"cycle_{key}_{rule}"
                    grouped = record.groupby(level="customer_id")
                    parts[label + "_closest"] = grouped.center.min()
                    parts[label + "_eligible"] = due.between(0, duration-1).groupby(level="customer_id").sum()
                    first = record.dropna(subset=["center"]).sort_values("center").groupby(level="customer_id").first()
                    parts[label + "_support"] = first.n
                    parts[label + "_due"] = first.due
            if key == "customer":
                for span in (365, 730):
                    parts[f"cycle_anniversary_{span}"] = (visits.age - span + duration/2).abs().groupby(visits.customer_id).min()
        block = pd.DataFrame(parts).reindex(customers.customer_id)
        block.index = customers.index
        blocks.append(block)
    return pd.concat(blocks).reindex(cohort.index)

extra = extra.join(cycle_features(train))
extra_test = extra_test.join(cycle_features(test))

In [ ]:
def sequence_features(cohort):
    blocks = []
    for cutoff, clients in cohort.groupby("scoring_date"):
        h = orders[orders.customer_id.isin(clients.customer_id) & (orders.order_date < cutoff)].copy()
        h["age"] = (cutoff - h.order_date).dt.days
        b = h.groupby(["customer_id", "order_id"]).agg(age=("age","first"), spend=("paid_price","sum"),
                items=("product_id","size"), loyalty=("loyalty_percent","mean")).reset_index()
        b = b.sort_values(["age","order_id"])
        g = b.groupby("customer_id")
        b["i"] = g.cumcount()
        b["gap"] = g.age.diff()
        parts = []
        for field in ["age","gap","spend","items","loyalty"]:
            parts.append(b[b.i < 8].pivot(index="customer_id", columns="i", values=field).add_prefix("seq_" + field + "_"))
        rec = g.age.first()
        median = g.gap.median()
        info = {}
        for name, period in [("typical", median), ("recent", g.head(4).groupby("customer_id").gap.median())]:
            period = period.replace(0, np.nan)
            info["seq_next_" + name] = np.ceil(rec / period) * period - rec
            for k in [1, 2, 3]:
                info[f"seq_center_{name}_{k}"] = (period * k - rec - 45).abs()
        for key in ["category","category2","gender_product","brand"]:
            sub = h.drop_duplicates(["customer_id",key,"order_id"]).sort_values("age")
            gs = sub.groupby(["customer_id",key])
            sub["gap"] = gs.age.diff()
            stats = gs.agg(n=("age","size"),last=("age","first"),period=("gap","median"),amount=("paid_price","sum"))
            stats["due"] = stats.period - stats["last"]
            for priority in ("n","amount","last"):
                main = stats.sort_values(priority, ascending=(priority=="last"), kind="stable").groupby(level=0).head(1).droplevel(1)
                info[f"seq_{key}_{priority}_due"] = main.due
                info[f"seq_{key}_{priority}_distance"] = (main.due - 45).abs()
        parts.append(pd.DataFrame(info))
        block = pd.concat(parts,axis=1).reindex(clients.customer_id)
        block.index = clients.index
        blocks.append(block)
    return pd.concat(blocks).reindex(cohort.index)

extra = extra.join(sequence_features(train))
extra_test = extra_test.join(sequence_features(test))


### Категории и переходы
Для заказа из нескольких категорий сравниваем варианты выбора основной категории по стоимости и числу товарных позиций. Из последовательности заказов получаем интервал до следующей покупки после похожей корзины и изменения активности по кварталам.

In [ ]:
def category_clocks(cohort):
    blocks = []
    for cutoff, clients in cohort.groupby("scoring_date"):
        h = orders.loc[orders.customer_id.isin(clients.customer_id) & (orders.order_date < cutoff)].copy()
        h["age"] = (cutoff - h.order_date).dt.days
        cats = h.groupby(["customer_id","order_id","category"]).agg(age=("age","first"), amount=("paid_price","sum"), lines=("product_id","size")).reset_index()
        parts = []
        for variant, baskets in [
            ("any", cats),
            ("value", cats.sort_values(["amount","lines"], ascending=False).drop_duplicates("order_id")),
            ("volume", cats.sort_values(["lines","amount"], ascending=False).drop_duplicates("order_id")),
            ("first", h.drop_duplicates("order_id")),
            ("expensive",h.sort_values("paid_price",ascending=False).drop_duplicates("order_id")),
        ]:
            b = baskets.sort_values("age").copy()
            g = b.groupby(["customer_id","category"])
            b["gap"] = g.age.diff()
            a = g.agg(age=("age","first"), median=("gap","median"), mean=("gap","mean"), n=("age","size"))
            a["due"] = a["median"] - a.age
            a["center"] = (a.due - 45).abs()
            a["due_mean"] = a["mean"] - a.age
            for field in ["age","median","n","due","center","due_mean"]:
                parts.append(a[field].unstack().add_prefix(f"clock_{variant}_{field}_"))
            recent = a.sort_values("age").groupby(level=0).head(1).droplevel(1)
            top = a.sort_values(["n","age"],ascending=[False,True]).groupby(level=0).head(1).droplevel(1)
            for name, stats in [("recent",recent),("top",top)]:
                parts.append(stats[["due","center","n","age"]].add_prefix(f"clock_{variant}_{name}_"))
        block = pd.concat(parts,axis=1).reindex(clients.customer_id)
        block.index = clients.index
        blocks.append(block)
    return pd.concat(blocks).reindex(cohort.index)

clock = category_clocks(train)
clock_test = category_clocks(test).reindex(columns=clock.columns)


In [ ]:
def transitions(cohort):
    blocks = []
    for cutoff, clients in cohort.groupby("scoring_date"):
        h = orders.loc[orders.customer_id.isin(clients.customer_id) & (orders.order_date < cutoff)].copy()
        h["age"] = (cutoff - h.order_date).dt.days
        base = h.groupby(["customer_id","order_id"]).age.first().reset_index().sort_values(["customer_id","age","order_id"])
        history = base.groupby("customer_id")
        base["next_gap"] = base.age - history.age.shift(1)
        parts = []
        for key in ["category","category2","gender_product","brand"]:
            items = h.groupby(["customer_id","order_id",key]).agg(value=("paid_price","sum"),count=("paid_price","size")).reset_index()
            main = items.sort_values(["count","value"],ascending=False).drop_duplicates("order_id")[["order_id",key]]
            receipts = base.merge(main,on="order_id",validate="one_to_one")
            g = receipts.groupby("customer_id")
            receipts["next_key"] = g[key].shift(1)
            receipts["previous_key"] = g[key].shift(-1)
            receipts["same"] = receipts[key] == receipts.next_key
            latest = g.first()
            # Match a past basket to the most recent basket, then measure the wait to its successor.
            for match, selected in [
                ("same",receipts[key] == receipts.customer_id.map(latest[key])),
                ("pair",(receipts[key] == receipts.customer_id.map(latest[key])) &
                        (receipts.previous_key == receipts.customer_id.map(latest.previous_key))),
                ("change",~receipts.same),
                ("repeat",receipts.same)
            ]:
                gaps = receipts[selected].groupby("customer_id").next_gap.agg(["median","mean","first","size"])
                for col in ["median","mean","first"]:
                    gaps[col] -= latest.age
                parts.append(gaps.add_prefix(f"path_{key}_{match}_"))
            # Project the most frequent successor of the current category.
            matches = receipts[receipts[key] == receipts.customer_id.map(latest[key])]
            choices = matches.dropna(subset=["next_key"]).groupby(["customer_id","next_key"]).next_gap.agg(["median","size"]).reset_index()
            choice = choices.sort_values("size",ascending=False).drop_duplicates("customer_id").set_index("customer_id")
            parts.append((choice["median"] - latest.age).rename(f"path_{key}_successor"))
        block = pd.concat(parts,axis=1).reindex(clients.customer_id)
        block.index = clients.index
        blocks.append(block)
    return pd.concat(blocks).reindex(cohort.index)


In [ ]:
def basket_patterns(cohort):
    blocks = []
    for cutoff, clients in cohort.groupby("scoring_date"):
        h = orders.loc[orders.customer_id.isin(clients.customer_id) & (orders.order_date < cutoff)].copy()
        h["age"] = (cutoff - h.order_date).dt.days
        h["age_bin"] = (h.age-14)//90
        agg = h.groupby(["customer_id","age_bin"]).agg(spend=("paid_price","sum"),orders=("order_id","nunique"),
                                                       items=("paid_price","size"), loyalty=("loyalty_percent","mean"))
        parts = []
        for field in agg:
            wide = agg[field].unstack(fill_value=0)
            parts.append(wide.add_prefix("pattern_" + field + "_q"))
            parts.append(((wide[0]+1)/(wide[1]+1)).rename("pattern_trend_"+field))
        for key in ["category","gender_product"]:
            items = h.groupby(["customer_id","order_id",key]).agg(value=("paid_price","sum"),lines=("paid_price","size"),age=("age","first")).reset_index()
            b = items.sort_values(["lines","value"],ascending=False).drop_duplicates("order_id").sort_values(["age","order_id"])
            g = b.groupby("customer_id")
            b["i"] = g.cumcount()
            b["code"] = pd.Categorical(b[key],categories=sorted(orders[key].unique())).codes
            width = orders[key].nunique()+1
            w = b[b.i<8].pivot(index="customer_id",columns="i",values="code").fillna(-1)+1
            parts.append(w.add_prefix("pattern_"+key+"_position_"))
            for k in [2,3,4]:
                code = sum(w[j]*width**j for j in range(k))
                parts.append(code.rename(f"pattern_{key}_last{k}"))
            for lag in range(1,5):
                b["match"] = b[key].eq(g[key].shift(-lag)) & g[key].shift(-lag).notna()
                b["when"] = b.age.where(b.match)
                parts.append(g.match.sum().rename(f"pattern_{key}_repeats_{lag}"))
                parts.append(g.when.min().rename(f"pattern_{key}_recency_{lag}"))
            b["prev"] = g[key].shift(-1)
            b["switch"] = b[key].ne(b.prev) & b.prev.notna()
            switches = b[b.switch].copy()
            gs = switches.groupby("customer_id")
            switches["interval"] = gs.age.diff()
            parts.append((gs.interval.median()-gs.age.min()).rename(f"pattern_{key}_switch_due"))
            parts.append(gs.size().rename(f"pattern_{key}_switch_count"))
            parts.append(gs.age.min().rename(f"pattern_{key}_switch_age"))
        block=pd.concat(parts,axis=1).reindex(clients.customer_id)
        block.index=clients.index
        blocks.append(block)
    return pd.concat(blocks).reindex(cohort.index)


In [ ]:
def forecast_windows(features):
    result = {}
    typical = features.recency - features.rhythm_order_all_median_delta
    due = typical - features.recency
    center = (due - 45).abs()
    result["window_typical_center"] = center
    for variant in ["volume", "value", "first"]:
        prefix = "clock_" + variant
        purity = features[prefix+"_top_n"]/features.n_orders
        result["window_"+variant+"_purity"] = purity
        for threshold in [0.75, 0.999]:
            eligible = (features.n_orders <= 3) | (purity >= threshold)
            result[f"window_{variant}_{threshold}"] = center.where(eligible)
        for which in ["recent","top"]:
            c = features[prefix+"_"+which+"_due"]
            count = features[prefix+"_"+which+"_n"]
            for support in [2,3,4]:
                blended = c.where((count>=support)&(features.n_orders>=4), due)
                result[f"window_{variant}_{which}_{support}"] = (blended - 45).abs()
    result["window_typical_band"] = center.where(center<=60)
    for k in [2,3,4]:
        result[f"window_position_{k}"] = ((k*typical-features.recency)-45).abs()
    return pd.DataFrame(result)


In [ ]:
def assemble(base, history, category, cohort):
    result = base.join(history).join(category)
    result = result.join(transitions(cohort)).join(basket_patterns(cohort))
    return result.join(forecast_windows(result))

F = assemble(X, extra, clock, train)
T = assemble(X_test, extra_test, clock_test, test).reindex(columns=F.columns)
assert F.columns.is_unique and F.index.equals(y.index)
print(f"{len(F.columns)} признаков")

### Предыдущее решение
Общий бустинг, три модели по группам клиентов и ранжировщик образуют базу для сравнения.

In [ ]:
def cadence_groups(frame):
    pure = frame.clock_volume_top_n.eq(frame.n_orders)
    return np.select([frame.n_orders.eq(1), frame.n_orders.le(3) | pure], [0, 1], default=2)

def segmented_predict(x, labels, query):
    groups = cadence_groups(x)
    other_groups = cadence_groups(query)
    scores = np.zeros(len(query))
    for group_id in range(3):
        cols = list(x.columns)
        if group_id == 1:
            cols = [c for c in cols if c.startswith(("window_", "rhythm_order_all_", "seq_age_"))]
            cols += ["n_orders", "n_days", "recency", "total_paid", "orders_180", "mean_price", "items_90"]
        elif group_id == 0:
            cols = ["age_days", "total_paid", "mean_price", "n_items", "mean_discount",
                    "mean_loyalty"] + list(x.filter(like="mix_").columns)
        mask = groups == group_id
        held = other_groups == group_id
        model = HistGradientBoostingClassifier(max_iter=150,learning_rate=.055,max_leaf_nodes=15,
                    min_samples_leaf=45,l2_regularization=6,early_stopping=False,random_state=42)
        with threadpool_limits(limits=4):
            model.fit(x.loc[mask, cols], labels.loc[mask])
            scores[held] = model.predict_proba(query.loc[held, cols])[:,1]
    return scores


In [ ]:
def fit_ranker(x, labels, dates):
    from lightgbm import LGBMRanker
    rng = np.random.default_rng(42)
    rows, sizes = [], []
    for date in np.unique(dates):
        indices = rng.permutation(np.flatnonzero(np.asarray(dates)==date))
        for query in np.array_split(indices, max(1, len(indices)//100)):
            rows.extend(query)
            sizes.append(len(query))
    model = LGBMRanker(n_estimators=300, num_leaves=15, learning_rate=.04, max_bin=127,
                       min_child_samples=80, colsample_bytree=.65, reg_lambda=5, random_state=42,
                       n_jobs=4, verbosity=-1, lambdarank_truncation_level=23)
    # NumPy inputs keep arbitrary category names out of the LightGBM feature-name format.
    return model.fit(x.iloc[rows].to_numpy(),labels.iloc[rows],group=sizes)


In [ ]:
def predictions(fit, labels, dates, held):
    classifier = fit_model(fit, labels)
    with threadpool_limits(limits=4):
        general = classifier.predict_proba(held)[:, 1]
    specialist = segmented_predict(fit, labels, held)
    mixture = (general + specialist) / 2
    ranker = fit_ranker(fit, labels, dates)
    raw = ranker.predict(held.to_numpy())
    ranked = np.interp(pd.Series(raw).rank(pct=True), np.linspace(0, 1, len(raw)), np.sort(mixture))
    return {"Общая модель": general, "Специалисты": specialist,
            "Смесь классификаторов": mixture, "Ранжировщик": ranked,
            **{f"Смесь с ранжировщиком {weight:.0%}": (1-weight)*mixture + weight*ranked
               for weight in [0.15, 0.35, 0.60]}}

### Представление чека и переходы
Каждый заказ получает основную категорию по числу товарных позиций. При равенстве сравниваем потраченные суммы. История переходов показывает, какую категорию клиент обычно покупал после последней категории или пары категорий.

In [ ]:
def receipt_table(orders, cohort):
    h = orders.merge(cohort[["customer_id","scoring_date"]], on="customer_id",validate="many_to_one")
    h = h.loc[h.order_date < h.scoring_date].copy()
    h["age"] = (h.scoring_date-h.order_date).dt.days
    ids=["customer_id","order_id"]
    receipts=h.groupby(ids,sort=False).agg(age=("age","first"), amount=("paid_price","sum"),
        units=("paid_price","size"), discount=("discount_percent","mean"),
        loyalty=("loyalty_percent","mean"), variety=("category","nunique"))
    counts=h.groupby(ids+["category"],sort=False).agg(lines=("paid_price","size"),value=("paid_price","sum")).reset_index()
    main=counts.sort_values(["lines","value","category"],ascending=[False,False,True]).drop_duplicates(ids)
    receipts=receipts.join(main.set_index(ids).category).reset_index()
    return receipts.sort_values(["customer_id","age","order_id"],ascending=[True,False,True]).reset_index(drop=True)



In [ ]:
def transition_clocks(receipts):
    b = receipts.copy()
    g = b.groupby("customer_id")
    next_cat = g.category.shift(-1)
    gap = b.age - g.age.shift(-1)
    b["prev"] = g.category.shift(1)
    b["next"] = next_cat
    b["gap"] = gap
    last_cat = g.category.last()
    current = b.customer_id.map(last_cat)
    previous_at_end = b[b.groupby("customer_id").cumcount(ascending=False)==1].set_index("customer_id").category
    # One row per customer and candidate category. Keep the timing of returns
    # to that category separate from the delay after leaving the current one.
    cg = b.groupby(["customer_id","category"])
    b["return_gap"] = -cg.age.diff()
    a=cg.agg(age=("age","last"), count=("age","size"), median=("return_gap","median"),
             first_age=("age","first"), last_gap=("return_gap","last"))
    a["global_recency"]=g.age.last().reindex(a.index.get_level_values(0)).values
    global_gap = g.gap.median().reindex(a.index.get_level_values(0)).to_numpy()
    a["personal_due"] = global_gap - a.global_recency
    a["category_due"] = a["median"].fillna(pd.Series(global_gap,index=a.index))-a.age
    a["same_anchor_due"]=a["median"].fillna(pd.Series(global_gap,index=a.index))-a.global_recency
    a["global_anchor_due"] = global_gap - a.age
    out={}
    for context, match in [("one", b.category.eq(current)),
                            ("two", b.category.eq(current)&b.prev.eq(b.customer_id.map(previous_at_end)))]:
        hist=b.loc[match & b.next.notna()].groupby(["customer_id","next"]).gap.agg(["size","median","last"])
        hist.index.names=a.index.names
        h=a.join(hist,rsuffix="_after")
        h["mass"]=h["size"].fillna(0)+h["count"]/h["count"].groupby(level=0).transform("sum")
        h["mass"]/=h.mass.groupby(level=0).transform("sum")
        h["transition_due"]=h.median_after - h.global_recency
        h["blended_due"]=h.transition_due.fillna(h.category_due)
        for c in ["category_due","same_anchor_due","global_anchor_due","blended_due"]:
            center=(h[c]-45).abs()
            prob=np.exp(-0.5*(center/32)**2)
            k=f"transition_{context}_{c}"
            out[k+"_prob"]=(prob*h.mass).groupby(level=0).sum()
            out[k+"_best"]=(prob*h.mass).groupby(level=0).max()
            best=h.sort_values(["mass","count"],ascending=False).groupby(level=0).head(1).droplevel(1)
            out[k+"_mode"]=best[c]
            out[k+"_expected"]=(h[c]*h.mass).groupby(level=0).sum()
        out["transition_"+context+"_support"]=h["size"].groupby(level=0).sum()
    return pd.DataFrame(out)


### Смена категорий и индивидуальные серии заказов
Считаем возвраты в прежнюю категорию, новые переходы и длину непрерывных серий по последним заказам и календарным окнам. Для каждого клиента делим историю на серии двумя способами — по медианному интервалу и по наибольшему разрыву между отсортированными логарифмами интервалов.

In [ ]:
def basket_graph(b):
    b=b.copy()
    g=b.groupby("customer_id")
    b["prior"]=g.category.shift()
    b["older"]=g.category.shift(2)
    b["idx"]=g.cumcount(ascending=False)
    b["change"]=b.prior.notna() & b.category.ne(b.prior)
    b["bounce"]=b.change & b.older.eq(b.category)
    b["novel"]=~b.duplicated(["customer_id","category"])
    b["pair"]=b.prior.fillna("START")+"→"+b.category
    b["rare_pair"]=~b.duplicated(["customer_id","pair"]) & b.prior.notna()
    b["run"]=(b.change).groupby(b.customer_id).cumsum()
    b["gap"]=-g.age.diff()
    parts=[]
    for name,mask in [("last4",b.idx<4),("last8",b.idx<8),("month",b.age<44),
                      ("quarter",b.age<104),("half",b.age<194),("history",b.age>=0)]:
        sample=b[mask]
        grouped=sample.groupby("customer_id")
        a=grouped.agg(count=("age","size"),states=("category","nunique"),edges=("pair","nunique"),
             switches=("change","sum"),returns=("bounce","sum"),new_states=("novel","sum"),
             first_edges=("rare_pair","sum"),spend=("amount","sum"),gap=("gap","median"))
        a["alternation"]=a.returns/a.switches.clip(lower=1)
        a["repeat_share"]=1-a.edges/a["count"]
        a["switch_share"]=a.switches/a["count"]
        runs=sample.groupby(["customer_id","run"]).size()
        a["longest_run"]=runs.groupby(level=0).max()
        parts.append(a.add_prefix(f"graph_{name}_"))
    for event in ["change","bounce","novel","rare_pair"]:
        sub=b[b[event]].copy()
        grouped=sub.groupby("customer_id")
        sub["delay"]=-grouped.age.diff()
        a=grouped.agg(age=("age","last"),n=("age","size"),period=("delay","median"))
        a["due"]=a.period-a.age
        parts.append(a.add_prefix(f"graph_event_{event}_"))
    return pd.concat(parts,axis=1)


In [ ]:
def personal_sessions(receipts):
    b=receipts.copy()
    g=b.groupby("customer_id")
    b["gap"]=-g.age.diff()
    b["log_gap"]=np.log1p(b.gap)
    ordered=b[b.gap.notna()].sort_values(["customer_id","log_gap"])
    ordered["separation"]=ordered.groupby("customer_id").log_gap.diff()
    cut=ordered.dropna(subset=["separation"]).sort_values("separation",ascending=False).drop_duplicates("customer_id")
    boundary=np.expm1(cut.set_index("customer_id").eval("log_gap - separation / 2"))
    out=[]
    for name,threshold in [("largest_break",boundary),("median_wait",g.gap.median())]:
        b["break"]=b.gap.gt(b.customer_id.map(threshold)) | b.gap.isna()
        b["session"]=g["break"].cumsum()
        sessions=b.groupby(["customer_id","session"]).agg(begin=("age","max"),end=("age","min"),
                   size=("age","size"),value=("amount","sum"),cats=("category","nunique"))
        sg=sessions.groupby(level=0)
        sessions["idle"]=sg.end.shift()-sessions.begin
        sessions["period"]=-sg.begin.diff()
        stats=sg.agg(sessions=("size","size"),last_size=("size","last"),normal_size=("size","median"),
                     last_variety=("cats","last"),normal_variety=("cats","mean"),last_value=("value","last"),
                     typical_rest=("idle","median"),last_rest=("idle","last"),typical_period=("period","median"))
        stats["boundary"]=threshold
        stats["size_ratio"]=stats.last_size/stats.normal_size
        stats["due_from_end"]=stats.typical_rest-sg.end.last()
        stats["due_from_start"]=stats.typical_period-sg.begin.last()
        b["state"]=b["break"].astype(int)
        b["following"]=-g.age.shift(-1)+b.age
        last_state=g.state.last()
        cases=b[b.state.eq(b.customer_id.map(last_state))]
        extrap=cases.groupby("customer_id").following.agg(["median","mean","size"])
        extrap["due"]=extrap["median"]-g.age.last()
        out.append(stats.join(extrap).add_prefix(f"session_{name}_"))
    return pd.concat(out,axis=1)


In [ ]:
def extend_features(frame, cohort):
    receipts = receipt_table(orders, cohort)
    def align(stats):
        return stats.reindex(cohort.customer_id).set_axis(cohort.index)
    transitions_frame = frame.join(align(transition_clocks(receipts)))
    full_frame = transitions_frame.join(align(basket_graph(receipts))).join(align(personal_sessions(receipts)))
    return transitions_frame, full_frame

transition_train, full_train = extend_features(F, train)
transition_test, full_test = extend_features(T, test)
transition_test = transition_test.reindex(columns=transition_train.columns)
full_test = full_test.reindex(columns=full_train.columns)
print("Переходы и все признаки:", transition_train.shape, full_train.shape)

### Региональные модели
Сначала отбираем признаки по уменьшению функции потерь на обучающих клиентах. Затем дополняем общий набор признаками каждой группы. Раздельно рассматриваем клиентов с одним заказом, с короткой или однородной историей и остальных клиентов. В последней группе учитываем, попадает ли прогнозная дата в интервал от −14 до +104 дней от даты скоринга.

In [ ]:
def select_by_gain(x,labels,limit=96):
    model=fit_model(x,labels)
    gain=np.zeros(x.shape[1])
    for stage in model._predictors:
        nodes=stage[0].nodes
        splits=nodes[nodes["is_leaf"]==0]
        np.add.at(gain,splits["feature_idx"],splits["gain"])
    return x.columns[np.argsort(-gain,kind="stable")[:limit]]

In [ ]:
def calendar_groups(frame):
    single = frame.n_orders.eq(1)
    regular = frame.n_orders.le(3) | frame.clock_volume_top_n.eq(frame.n_orders)
    due = -frame.rhythm_order_all_median_delta
    return np.select([single, regular, due.between(-14, 104)], [0, 1, 2], default=3)

def regional_forecast(fit, labels, query):
    shared = select_by_gain(fit, labels, 96)
    region_fit = calendar_groups(fit)
    region_query = calendar_groups(query)
    hist = np.empty(len(query))
    for region in range(4):
        rows = region_fit == region
        held = region_query == region
        x = fit.loc[rows]
        y = labels.loc[rows]
        columns = shared.union(select_by_gain(x, y, 60), sort=False)
        x = x[columns]
        q = query.loc[held, columns]
        model = HistGradientBoostingClassifier(max_iter=160, max_leaf_nodes=12, min_samples_leaf=40,
                    l2_regularization=6, early_stopping=False, learning_rate=.05, random_state=42)
        with threadpool_limits(limits=4):
            model.fit(x, y)
            hist[held] = model.predict_proba(q)[:, 1]
    return hist

### Сохранение этапов
Ключ контрольной точки учитывает матрицу признаков, идентификаторы и метки, версии библиотек и исполняемый код моделей. Для обучающего и тестового прогнозов создаются отдельные записи. Отчёт по валидации записывается после каждого месяца.

In [ ]:
import hashlib
import inspect
import os
import tempfile
from importlib.metadata import version

def checkpoint_key(frame, cohort):
    digest = hashlib.sha256()
    for obj in (frame, cohort):
        digest.update(repr(list(zip(obj.columns, obj.dtypes.astype(str)))).encode())
        digest.update(pd.util.hash_pandas_object(obj, index=True).to_numpy().tobytes())
    for fn in (fit_model, fit_ranker, cadence_groups, segmented_predict, predictions,
               select_by_gain, calendar_groups, regional_forecast):
        digest.update(inspect.getsource(fn).encode())
    for package in ("numpy", "pandas", "scikit-learn", "lightgbm"):
        digest.update(version(package).encode())
    return digest.hexdigest()[:24]

cache = Path(".checkpoints/lamoda_groups") / checkpoint_key(full_train, train)
cache.mkdir(parents=True, exist_ok=True)

def cached_forecast(name, ids, calculate):
    path = cache / (name + ".npz")
    if path.exists():
        with np.load(path, allow_pickle=False) as saved:
            scores = saved["scores"]
            if (np.array_equal(saved["ids"], ids) and scores.shape == (len(ids),)
                    and np.isfinite(scores).all() and ((0 <= scores) & (scores <= 1)).all()):
                return scores
    scores = np.asarray(calculate())
    assert scores.shape == (len(ids),) and np.isfinite(scores).all()
    assert ((0 <= scores) & (scores <= 1)).all()
    with tempfile.NamedTemporaryFile(dir=cache, delete=False) as tmp:
        np.savez_compressed(tmp, ids=np.asarray(ids), scores=scores)
    os.replace(tmp.name, path)
    return scores

def score_stage(stage, rows, ids, transitions_query, full_query):
    parts = {}
    for name, frame, query in [("transitions", transition_train, transitions_query),
                               ("full", full_train, full_query)]:
        parts[name] = cached_forecast(stage + "_" + name, ids, lambda: predictions(
            frame.loc[rows], y.loc[rows], train.scoring_date.loc[rows], query
        )["Смесь с ранжировщиком 60%"])
    local = cached_forecast(stage + "_regional", ids, lambda: regional_forecast(
        full_train.loc[rows], y.loc[rows], full_query))
    return {"transitions": parts["transitions"], "regional": local,
            "transitions_regional": (parts["transitions"] + local) / 2,
            "full_regional": (parts["full"] + local) / 2,
            "combined_regional": (parts["transitions"] + parts["full"] + 2 * local) / 4}

In [ ]:
results = []
for date in sorted(train.scoring_date.unique())[1:]:
    fit_rows = train.scoring_date < date
    held = train.scoring_date == date
    scores = score_stage(str(date.date()), fit_rows, train.customer_id.loc[held],
                         transition_train.loc[held], full_train.loc[held])
    for name, values in scores.items():
        k = int(np.ceil(len(values) * .2))
        top = np.argsort(-values, kind="stable")[:k]
        results.append({"date": str(date.date()), "model": name, "ap20": ap20(y.loc[held], values),
                        "buyers_in_top_5000": int(y.loc[held].iloc[top].sum())})
    report = pd.DataFrame(results)
    with tempfile.NamedTemporaryFile(mode="w", dir=".", delete=False) as tmp:
        report.to_csv(tmp, index=False)
    os.replace(tmp.name, "validation_groups.csv")
summary = report.pivot(index="model", columns="date", values="ap20")
summary["mean"] = summary.mean(axis=1)
best = summary["mean"].idxmax()
print(summary.sort_values("mean", ascending=False).round(5).to_string())
print("Выбрано по средней AP@20%:", best)

### Финальный прогноз
Веса выбираются по двум временным срезам. Эти же срезы использованы в исследовании, поэтому отдельной проверкой будет Kaggle.

In [ ]:
test_key = checkpoint_key(full_test, test)
final = score_stage("test_" + test_key, np.ones(len(train), dtype=bool), test.customer_id,
                    transition_test, full_test)[best]
assert sample.customer_id.is_unique and test.customer_id.is_unique
assert set(sample.customer_id) == set(test.customer_id)
submission = sample[["customer_id"]].copy()
submission["score"] = submission.customer_id.map(pd.Series(final, index=test.customer_id))
assert submission.score.between(0, 1).all()
submission.to_csv("submission_groups.csv", index=False)
pd.testing.assert_frame_equal(submission, pd.read_csv("submission_groups.csv"))
print(f"Проверены {len(submission):,} строк и порядок клиентов по образцу Kaggle.")